<div style="display:flex; justify-content:space-between; align-items:center; width:100%; margin:8px 0 24px 0;">
  <div style="text-align:left;">
    <img src="https://www.ec-nantes.fr/medias/photo/logocn-rvb_1648479844750-png?ID_FICHE=178994&amp;INLINE=FALSE" alt="Centrale Nantes" style="height:72px; width:auto;">
  </div>
  <div style="text-align:right; font-size:18px; font-weight:600; color:#17324d; line-height:1.35;">
    MSc. CORO DASSIP
  </div>
</div>

<div style="border:2px solid #333; padding:14px 20px; margin:15px auto 25px auto; width:85%; max-width:900px; box-sizing:border-box; text-align:center;">
  <h1 style="margin:0;"><b>Image Segmentation — Implementation</b></h1>
</div>

This notebook executes the classical segmentation experiments covering thresholding, morphology, connected components, contours, color segmentation, distance-transform/watershed separation, quantitative mask evaluation, and validation.


## Setup — Environment and Configuration

In [ ]:
# Import the numerical and image-processing tools used across the segmentation pipeline.
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

from scipy import ndimage

import cv2

np.set_printoptions(precision=3, suppress=True)

print("NumPy :", np.__version__)
print("OpenCV:", cv2.__version__)
print("Setup : PASS")

### 0.1 Locate the Lab Automatically


In [ ]:
def find_lab_root():
    """Locate the current Segmentation lab independent of launch directory.
    
    Returns
    -------
    Path
        Lab root containing hand.png and notebooks/main.ipynb.
    """
    cwd = Path.cwd().resolve()

    # Accept execution from either the lab root or its notebooks directory.
    for candidate in [cwd, *cwd.parents]:
        # Confirm a lab-specific sentinel file plus main notebook before accepting the root.
        if (
            (candidate / "data" / "hand.png").is_file()
            and (candidate / "notebooks" / "main.ipynb").is_file()
        ):
            return candidate

    raise FileNotFoundError(
        "Could not locate the segmentation lab root."
    )


LAB_ROOT = find_lab_root()
DATA_DIR = LAB_ROOT / "data"
OUTPUT_DIR = LAB_ROOT / "outputs" / "figures"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Lab root:", LAB_ROOT)
print("Data dir:", DATA_DIR)
print("Output  :", OUTPUT_DIR)

### 0.2 Reusable helpers

In [ ]:
# Centralize image I/O so segmentation operates on consistent dtype/channel conventions.
def load_gray(path):
    """Load one image under the grayscale segmentation contract.
    
    Parameters
    ----------
    path : path-like
        Image file to read.
    
    Returns
    -------
    ndarray
        uint8 grayscale image.
    """
    return np.asarray(
        Image.open(path).convert("L"),
        dtype=np.uint8
    )


def load_rgb(path):
    """Load one image while preserving RGB color information.
    
    Parameters
    ----------
    path : path-like
        Image file to read.
    
    Returns
    -------
    ndarray
        uint8 RGB image.
    """
    return np.asarray(
        Image.open(path).convert("RGB"),
        dtype=np.uint8
    )


def show_gray(ax, image, title):
    """Display one scalar mask or image consistently.
    
    Parameters
    ----------
    ax : matplotlib.axes.Axes
        Target axes.
    image : ndarray
        Scalar image/mask.
    title : str
        Plot title.
    
    Returns
    -------
    None
    """
    ax.imshow(image, cmap="gray")
    ax.set_title(title)
    ax.axis("off")


def show_rgb(ax, image, title):
    """Display one RGB image consistently.
    
    Parameters
    ----------
    ax : matplotlib.axes.Axes
        Target axes.
    image : ndarray
        RGB image.
    title : str
        Plot title.
    
    Returns
    -------
    None
    """
    ax.imshow(image)
    ax.set_title(title)
    ax.axis("off")


def save_figure(fig, filename):
    """Persist one segmentation diagnostic figure.
    
    Parameters
    ----------
    fig : matplotlib.figure.Figure
        Figure to save.
    filename : str
        Filename under outputs/figures/.
    
    Returns
    -------
    None
    """
    path = OUTPUT_DIR / filename
    fig.savefig(path, dpi=160, bbox_inches="tight")
    print("Saved:", path.name)


def to_uint8_mask(mask):
    """Convert arbitrary truth values to OpenCV mask encoding.
    
    Parameters
    ----------
    mask : array-like
        Boolean or binary-like mask.
    
    Returns
    -------
    ndarray
        uint8 mask containing only 0 and 255.
    
    Notes
    -----
    OpenCV morphology and connected-component functions expect 8-bit mask input.
    """
    return (
        np.asarray(mask).astype(bool) * 255
    ).astype(np.uint8)


def overlay_mask(image_rgb, mask, alpha=0.35):
    """Blend a red segmentation mask over an RGB image.
    
    Parameters
    ----------
    image_rgb : ndarray
        uint8 RGB source image.
    mask : array-like
        Foreground mask.
    alpha : float
        Overlay opacity; 0.35 by default.
    
    Returns
    -------
    ndarray
        uint8 RGB overlay.
    
    Notes
    -----
    A moderate alpha keeps source evidence visible so boundary quality can still be
    judged rather than hidden by the visualization.
    """
    output = image_rgb.astype(np.float32).copy()
    mask_bool = np.asarray(mask).astype(bool)

    red = np.zeros_like(output)
    red[..., 0] = 255

    output[mask_bool] = (
        (1 - alpha) * output[mask_bool]
        + alpha * red[mask_bool]
    )

    return np.clip(output, 0, 255).astype(np.uint8)

## 1. Segmentation Problem Formulation

## 2. Load the Lab Images


In [ ]:
# Load all benchmark images once and verify the intended grayscale/RGB roles.
hand = load_gray(
    DATA_DIR / "hand.png"
)

tower_rgb = load_rgb(
    DATA_DIR / "tower.jpg"
)

peppers_rgb = load_rgb(
    DATA_DIR / "peppers.png"
)

fig, axes = plt.subplots(
    1,
    3,
    figsize=(15, 5)
)

show_gray(
    axes[0],
    hand,
    "Hand"
)

show_rgb(
    axes[1],
    tower_rgb,
    "Tower"
)

show_rgb(
    axes[2],
    peppers_rgb,
    "Peppers"
)

fig.tight_layout()
save_figure(
    fig,
    "01_input_images.png"
)
plt.show()

print("hand shape   :", hand.shape)
print("tower shape  :", tower_rgb.shape)
print("peppers shape:", peppers_rgb.shape)

## 3. Histogram-Based Threshold Selection


In [ ]:
fig, axes = plt.subplots(
    1,
    2,
    figsize=(12, 4)
)

show_gray(
    axes[0],
    hand,
    "Hand image"
)

# Inspect the intensity distribution before selecting any thresholding strategy.
axes[1].hist(
    hand.ravel(),
    bins=256,
    range=(0, 255)
)
axes[1].set_title(
    "Hand intensity histogram"
)
axes[1].set_xlabel(
    "Intensity"
)
axes[1].set_ylabel(
    "Pixel count"
)

fig.tight_layout()
save_figure(
    fig,
    "02_hand_histogram.png"
)
plt.show()


> **Output comment.** The histogram reveals whether foreground and background intensities are sufficiently separated for global thresholding. Distinct modes support a single threshold, whereas broad overlap indicates that intensity alone may be insufficient and motivates smoothing, local thresholds, color cues, or region-based processing.


## 4. Manual Global Thresholding


In [ ]:
# Establish a transparent manual baseline before automatic threshold selection.
# Use one fixed illustrative baseline; later Otsu removes this manual decision.
manual_threshold = 120

mask_manual = hand < manual_threshold

fig, axes = plt.subplots(
    1,
    3,
    figsize=(15, 5)
)

show_gray(
    axes[0],
    hand,
    "Original"
)

show_gray(
    axes[1],
    mask_manual,
    f"Mask — T={manual_threshold}"
)

masked_hand = np.where(
    mask_manual,
    hand,
    255
)

show_gray(
    axes[2],
    masked_hand,
    "Segmented foreground"
)

fig.tight_layout()
save_figure(
    fig,
    "03_manual_threshold.png"
)
plt.show()


> **Output comment.** Manual thresholding provides a transparent baseline: every classification decision is determined by one intensity cutoff. Its simplicity makes the result easy to interpret, but it is sensitive to the selected value and assumes that the same threshold is appropriate across the entire image.


## 5. Threshold Sensitivity


In [ ]:
# Sweep plausible thresholds to reveal sensitivity of the foreground mask.
# Sweep values on both sides of the baseline to expose threshold sensitivity.
thresholds = [
    70,
    100,
    130,
    160
]

fig, axes = plt.subplots(
    1,
    len(thresholds),
    figsize=(16, 4)
)

for ax, threshold in zip(
    axes,
    thresholds
):
    mask = hand < threshold

    show_gray(
        ax,
        mask,
        f"T={threshold}"
    )

fig.tight_layout()
save_figure(
    fig,
    "04_threshold_sensitivity.png"
)
plt.show()


> **Output comment.** The threshold sweep shows how small parameter changes can alter foreground area, boundary position, and object connectivity. A stable segmentation should not collapse under a minor threshold perturbation; strong sensitivity indicates weak class separation or uneven illumination.


## 6. Otsu Thresholding


In [ ]:
# Invert because the hand is darker than the background.
otsu_threshold, mask_otsu_cv = cv2.threshold(
    hand,
    0,
    255,
    cv2.THRESH_BINARY_INV
    + cv2.THRESH_OTSU
)

mask_otsu = (
    mask_otsu_cv > 0
)

print(
    "Otsu threshold:",
    otsu_threshold
)

fig, axes = plt.subplots(
    1,
    3,
    figsize=(15, 5)
)

show_gray(
    axes[0],
    hand,
    "Original"
)

show_gray(
    axes[1],
    mask_otsu,
    f"Otsu mask — T={otsu_threshold:.1f}"
)

overlay = overlay_mask(
    np.stack([hand] * 3, axis=-1),
    mask_otsu
)

show_rgb(
    axes[2],
    overlay,
    "Mask overlay"
)

fig.tight_layout()
save_figure(
    fig,
    "05_otsu_threshold.png"
)
plt.show()


> **Output comment.** Otsu's method selects the threshold from the image histogram by maximizing between-class separation. It removes manual tuning when a meaningful two-class histogram exists, but it remains a global method and can fail when foreground/background distributions overlap strongly or illumination varies spatially.


## 7. Gaussian Smoothing Before Thresholding


In [ ]:
# Reduce local noise before Otsu to test whether class separation becomes more stable.
hand_blurred = cv2.GaussianBlur(
    # A 5x5 Gaussian removes local noise while preserving the hand boundary scale.
    hand,
    (5, 5),
    0
)

blur_otsu_threshold, blur_otsu_cv = cv2.threshold(
    hand_blurred,
    0,
    255,
    cv2.THRESH_BINARY_INV
    + cv2.THRESH_OTSU
)

blur_otsu = (
    blur_otsu_cv > 0
)

fig, axes = plt.subplots(
    1,
    3,
    figsize=(15, 5)
)

show_gray(
    axes[0],
    hand,
    "Original"
)

show_gray(
    axes[1],
    hand_blurred,
    "Gaussian-smoothed"
)

show_gray(
    axes[2],
    blur_otsu,
    "Otsu after smoothing"
)

fig.tight_layout()
save_figure(
    fig,
    "06_smoothing_before_otsu.png"
)
plt.show()


> **Output comment.** Pre-smoothing reduces small intensity fluctuations before threshold selection. This can stabilize the mask and suppress isolated noise, but excessive smoothing shifts boundaries and merges nearby structures. The comparison demonstrates that preprocessing directly changes the segmentation evidence available to the thresholding stage.


## 8. Adaptive Thresholding


In [ ]:
# Compare local thresholds under the same neighborhood and offset.
# A 31x31 neighborhood captures local illumination; C=5 rejects marginal dark noise.
adaptive_mean = cv2.adaptiveThreshold(
    hand,
    255,
    cv2.ADAPTIVE_THRESH_MEAN_C,
    cv2.THRESH_BINARY_INV,
    31,
    5
)

adaptive_gaussian = cv2.adaptiveThreshold(
    hand,
    255,
    cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
    cv2.THRESH_BINARY_INV,
    31,
    5
)

fig, axes = plt.subplots(
    1,
    3,
    figsize=(15, 5)
)

show_gray(
    axes[0],
    hand,
    "Original"
)

show_gray(
    axes[1],
    adaptive_mean,
    "Adaptive mean"
)

show_gray(
    axes[2],
    adaptive_gaussian,
    "Adaptive Gaussian"
)

fig.tight_layout()
save_figure(
    fig,
    "07_adaptive_thresholding.png"
)
plt.show()


> **Output comment.** Adaptive thresholding estimates the decision level locally, making it more robust to gradual illumination variation than one global cutoff. Its advantage is strongest when the same object appears under different local brightness conditions, although neighborhood size and offset become additional parameters that must be controlled.


## 9. Morphological Processing

## 10. Structuring Elements


In [ ]:
# Compare structuring-element geometry before applying morphology to the mask.
# Keep all candidate structuring elements at 7x7 so shape—not size—is compared.
# Keep all candidate elements at 7x7 so only geometry—not scale—changes.
kernel_rect = cv2.getStructuringElement(
    cv2.MORPH_RECT,
    (7, 7)
)

kernel_ellipse = cv2.getStructuringElement(
    cv2.MORPH_ELLIPSE,
    (7, 7)
)

kernel_cross = cv2.getStructuringElement(
    cv2.MORPH_CROSS,
    (7, 7)
)

fig, axes = plt.subplots(
    1,
    3,
    figsize=(10, 3)
)

show_gray(
    axes[0],
    kernel_rect,
    "Rectangle"
)

show_gray(
    axes[1],
    kernel_ellipse,
    "Ellipse"
)

show_gray(
    axes[2],
    kernel_cross,
    "Cross"
)

fig.tight_layout()
save_figure(
    fig,
    "08_structuring_elements.png"
)
plt.show()

## 11. Erosion and Dilation


In [ ]:
mask_uint8 = to_uint8_mask(
    blur_otsu
)

# Keep size fixed so the experiment isolates the morphological operator itself.
# Elliptical support reduces directional bias on rounded object boundaries.
kernel = cv2.getStructuringElement(
    cv2.MORPH_ELLIPSE,
    (7, 7)
)

eroded = cv2.erode(
    mask_uint8,
    kernel,
    iterations=1
)

dilated = cv2.dilate(
    mask_uint8,
    kernel,
    iterations=1
)

fig, axes = plt.subplots(
    1,
    3,
    figsize=(15, 5)
)

show_gray(
    axes[0],
    mask_uint8,
    "Original mask"
)

show_gray(
    axes[1],
    eroded,
    "Erosion"
)

show_gray(
    axes[2],
    dilated,
    "Dilation"
)

fig.tight_layout()
save_figure(
    fig,
    "09_erosion_dilation.png"
)
plt.show()


> **Output comment.** Erosion contracts foreground regions and removes small protrusions, while dilation expands them and can bridge narrow gaps. These are shape operations on the binary mask, so their effect depends on both the structuring-element geometry and the scale of the objects being processed.


## 12. Opening and Closing


In [ ]:
# Contrast opening and closing on exactly the same binary input and kernel.
opened = cv2.morphologyEx(
    mask_uint8,
    cv2.MORPH_OPEN,
    kernel
)

closed = cv2.morphologyEx(
    mask_uint8,
    cv2.MORPH_CLOSE,
    kernel
)

fig, axes = plt.subplots(
    1,
    3,
    figsize=(15, 5)
)

show_gray(
    axes[0],
    mask_uint8,
    "Original mask"
)

show_gray(
    axes[1],
    opened,
    "Opening"
)

show_gray(
    axes[2],
    closed,
    "Closing"
)

fig.tight_layout()
save_figure(
    fig,
    "10_opening_closing.png"
)
plt.show()


> **Output comment.** Opening removes small foreground structures while approximately preserving larger regions; closing fills narrow gaps and small holes. Their complementary behavior makes them useful for mask cleanup after thresholding, provided the structuring element is smaller than the structures that should be preserved.


## 13. Morphological Gradient


In [ ]:
# Extract a morphology-based boundary band without using image derivatives.
morph_gradient = cv2.morphologyEx(
    mask_uint8,
    cv2.MORPH_GRADIENT,
    kernel
)

fig, axes = plt.subplots(
    1,
    2,
    figsize=(10, 5)
)

show_gray(
    axes[0],
    mask_uint8,
    "Mask"
)

show_gray(
    axes[1],
    morph_gradient,
    "Morphological gradient"
)

fig.tight_layout()
save_figure(
    fig,
    "11_morphological_gradient.png"
)
plt.show()

## 14. Hole Filling


In [ ]:
# Fill only enclosed background regions without expanding the external boundary.
mask_with_holes = blur_otsu.astype(bool)

filled = ndimage.binary_fill_holes(
    mask_with_holes
)

fig, axes = plt.subplots(
    1,
    3,
    figsize=(15, 5)
)

show_gray(
    axes[0],
    mask_with_holes,
    "Before filling"
)

show_gray(
    axes[1],
    filled,
    "After filling"
)

show_gray(
    axes[2],
    filled.astype(int)
    - mask_with_holes.astype(int),
    "Pixels added"
)

fig.tight_layout()
save_figure(
    fig,
    "12_hole_filling.png"
)
plt.show()


> **Output comment.** Hole filling changes internal background regions enclosed by foreground into foreground without expanding the external object boundary. This is appropriate when objects are expected to be solid and internal gaps are segmentation artifacts rather than true background regions.


## 15. Connected Components


In [ ]:
# Convert the cleaned mask into measurable object-level regions.
# Use 8-connectivity so diagonally touching foreground pixels form one region.
num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(
    to_uint8_mask(filled),
    connectivity=8
)

print(
    "Number of foreground components:",
    num_labels - 1
)

component_areas = stats[
    1:,
    cv2.CC_STAT_AREA
]

print(
    "Foreground areas:",
    component_areas
)

fig, axes = plt.subplots(
    1,
    2,
    figsize=(12, 5)
)

show_gray(
    axes[0],
    filled,
    "Binary mask"
)

axes[1].imshow(
    labels,
    cmap="nipy_spectral"
)
axes[1].set_title(
    "Connected-component labels"
)
axes[1].axis("off")

fig.tight_layout()
save_figure(
    fig,
    "13_connected_components.png"
)
plt.show()


> **Output comment.** Connected-component analysis converts a binary mask into discrete labeled regions. Region count and area statistics expose fragmentation and small false positives, making the mask measurable at the object level rather than only at the pixel level.


## 16. Remove Small Components


In [ ]:
# Express noise rejection as an explicit region-area criterion.
# 500 px removes tiny artifacts while remaining far below the target hand area.
minimum_area = 500

clean_components = np.zeros_like(
    labels,
    dtype=bool
)

for label_id in range(
    1,
    num_labels
):
    area = stats[
        label_id,
        cv2.CC_STAT_AREA
    ]

    # Keep only components that satisfy the explicit object-size prior.
    if area >= minimum_area:
        clean_components |= (
            labels == label_id
        )

fig, axes = plt.subplots(
    1,
    2,
    figsize=(10, 5)
)

show_gray(
    axes[0],
    filled,
    "Before area filtering"
)

show_gray(
    axes[1],
    clean_components,
    f"Area ≥ {minimum_area}"
)

fig.tight_layout()
save_figure(
    fig,
    "14_component_area_filtering.png"
)
plt.show()


> **Output comment.** Area filtering removes regions that are too small to represent the target object under the chosen geometric assumptions. This is more interpretable than arbitrary pixel cleanup because the criterion is tied explicitly to component size; however, a threshold that is too large can delete legitimate small objects.


## 17. Contours


In [ ]:
# Convert cleaned connected regions into explicit boundary representations.
contours, hierarchy = cv2.findContours(
    to_uint8_mask(
        clean_components
    ),
    cv2.RETR_EXTERNAL,
    cv2.CHAIN_APPROX_SIMPLE
)

hand_rgb = np.stack(
    [hand] * 3,
    axis=-1
)

contour_view = hand_rgb.copy()

cv2.drawContours(
    contour_view,
    contours,
    -1,
    (255, 0, 0),
    2
)

fig, axes = plt.subplots(
    1,
    2,
    figsize=(10, 5)
)

show_gray(
    axes[0],
    clean_components,
    "Clean mask"
)

show_rgb(
    axes[1],
    contour_view,
    "Detected contours"
)

fig.tight_layout()
save_figure(
    fig,
    "15_contours.png"
)
plt.show()

print(
    "Number of external contours:",
    len(contours)
)

## 18. Region Properties


In [ ]:
# Measure geometry per contour so region filtering can use object-level evidence.
region_rows = []

for index, contour in enumerate(
    contours,
    start=1
):
    area = cv2.contourArea(
        contour
    )

    perimeter = cv2.arcLength(
        contour,
        True
    )

    x, y, w, h = cv2.boundingRect(
        contour
    )

    moments = cv2.moments(
        contour
    )

    # Centroid division is valid only for contours with non-zero spatial mass.
    if moments["m00"] != 0:
        cx = moments["m10"] / moments["m00"]
        cy = moments["m01"] / moments["m00"]
    else:
        cx = np.nan
        cy = np.nan

    circularity = (
        4 * np.pi * area
        / (perimeter ** 2)
        # Compactness is undefined for zero-perimeter degenerate contours.
        if perimeter > 0
        else np.nan
    )

    aspect_ratio = (
        w / h
        # Aspect ratio requires non-zero bounding-box height.
        if h > 0
        else np.nan
    )

    region_rows.append(
        {
            "component": index,
            "area": area,
            "perimeter": perimeter,
            "cx": cx,
            "cy": cy,
            "width": w,
            "height": h,
            "aspect_ratio": aspect_ratio,
            "circularity": circularity,
        }
    )

for row in region_rows:
    print(row)

## 19. Color Segmentation


In [ ]:
# Separate hue from brightness before defining chromatic segmentation rules.
peppers_hsv = cv2.cvtColor(
    peppers_rgb,
    cv2.COLOR_RGB2HSV
)

hue = peppers_hsv[..., 0]
saturation = peppers_hsv[..., 1]
value = peppers_hsv[..., 2]

fig, axes = plt.subplots(
    1,
    4,
    figsize=(16, 4)
)

show_rgb(
    axes[0],
    peppers_rgb,
    "RGB"
)

show_gray(
    axes[1],
    hue,
    "Hue"
)

show_gray(
    axes[2],
    saturation,
    "Saturation"
)

show_gray(
    axes[3],
    value,
    "Value"
)

fig.tight_layout()
save_figure(
    fig,
    "16_hsv_channels.png"
)
plt.show()

### HSV Range Experiment


In [ ]:
# Red wraps around the HSV hue axis, so two intervals are required.
# Saturation/value floors reject pale or very dark pixels that are not reliably red.
# Hue bounds target red near both ends of OpenCV's circular 0–179 hue range.
lower_red_1 = np.array(
    [0, 80, 50],
    dtype=np.uint8
)

upper_red_1 = np.array(
    [12, 255, 255],
    dtype=np.uint8
)

lower_red_2 = np.array(
    [165, 80, 50],
    dtype=np.uint8
)

upper_red_2 = np.array(
    [179, 255, 255],
    dtype=np.uint8
)

mask_red_1 = cv2.inRange(
    peppers_hsv,
    lower_red_1,
    upper_red_1
)

mask_red_2 = cv2.inRange(
    peppers_hsv,
    lower_red_2,
    upper_red_2
)

red_mask = (
    (mask_red_1 > 0)
    | (mask_red_2 > 0)
)

red_segment = peppers_rgb.copy()
red_segment[~red_mask] = 0

fig, axes = plt.subplots(
    1,
    3,
    figsize=(15, 5)
)

show_rgb(
    axes[0],
    peppers_rgb,
    "Original peppers"
)

show_gray(
    axes[1],
    red_mask,
    "Red-color mask"
)

show_rgb(
    axes[2],
    red_segment,
    "Segmented red regions"
)

fig.tight_layout()
save_figure(
    fig,
    "17_color_segmentation.png"
)
plt.show()


> **Output comment.** HSV-based segmentation separates chromatic information from brightness more directly than raw RGB thresholds. The red-mask experiment shows how color can provide discrimination when grayscale intensities overlap, while also demonstrating the need to handle hue wrap-around for colors near the ends of the hue scale.


## 20. Edge-Based Segmentation


In [ ]:
tower_gray = cv2.cvtColor(
    tower_rgb,
    cv2.COLOR_RGB2GRAY
)

# Pre-smooth the tower so Canny responds to structural edges rather than pixel noise.
tower_blur = cv2.GaussianBlur(
    tower_gray,
    (5, 5),
    0
)

# Use a 1:2 hysteresis ratio to keep strong edges and supported weak edges.
tower_edges = cv2.Canny(
    tower_blur,
    80,
    160
)

# A 5x5 closing kernel bridges short gaps without merging distant structures.
edge_kernel = cv2.getStructuringElement(
    cv2.MORPH_RECT,
    (5, 5)
)

# Two closing passes connect fragmented edge segments while limiting thickening.
tower_edges_closed = cv2.morphologyEx(
    tower_edges,
    cv2.MORPH_CLOSE,
    edge_kernel,
    iterations=2
)

fig, axes = plt.subplots(
    1,
    3,
    figsize=(15, 5)
)

show_rgb(
    axes[0],
    tower_rgb,
    "Tower"
)

show_gray(
    axes[1],
    tower_edges,
    "Canny edges"
)

show_gray(
    axes[2],
    tower_edges_closed,
    "Closed edge map"
)

fig.tight_layout()
save_figure(
    fig,
    "18_edge_based_segmentation.png"
)
plt.show()


> **Output comment.** Edges locate rapid intensity changes rather than homogeneous regions. They are useful for boundary evidence, but edge maps are often fragmented and do not directly produce filled objects. Additional linking, morphology, or region reasoning is therefore usually required before an edge-based result becomes a usable segmentation mask.


## 21. Distance Transform


In [ ]:
# Interior distance peaks provide markers for separating touching regions.
# A 5x5 Euclidean mask gives smoother distance estimates than the 3x3 option.
distance = cv2.distanceTransform(
    to_uint8_mask(clean_components),
    cv2.DIST_L2,
    5
)

fig, axes = plt.subplots(
    1,
    2,
    figsize=(10, 5)
)

show_gray(
    axes[0],
    clean_components,
    "Binary mask"
)

im = axes[1].imshow(
    distance,
    cmap="viridis"
)

axes[1].set_title(
    "Distance transform"
)

axes[1].axis("off")

fig.colorbar(
    im,
    ax=axes[1],
    fraction=0.046
)

fig.tight_layout()
save_figure(
    fig,
    "19_distance_transform.png"
)
plt.show()


> **Output comment.** The distance transform assigns each foreground pixel its distance to the nearest background pixel. Local maxima therefore correspond to deep interior object regions and provide useful markers for separating touching objects rather than merely describing the binary mask.


## 22. Watershed Segmentation


In [ ]:
watershed_input = peppers_rgb.copy()

peppers_gray = cv2.cvtColor(
    peppers_rgb,
    cv2.COLOR_RGB2GRAY
)

_, peppers_binary = cv2.threshold(
    peppers_gray,
    0,
    255,
    cv2.THRESH_BINARY
    + cv2.THRESH_OTSU
)

# A 3x3 kernel keeps marker cleanup local and limits geometric drift.
ws_kernel = np.ones(
    (3, 3),
    np.uint8
)

opening = cv2.morphologyEx(
    peppers_binary,
    cv2.MORPH_OPEN,
    ws_kernel,
    iterations=2
)

# Extra dilation builds a conservative region known to be background.
sure_bg = cv2.dilate(
    opening,
    ws_kernel,
    # Three background dilations deliberately create a conservative sure-background region.
iterations=3
)

dist_transform = cv2.distanceTransform(
    opening,
    cv2.DIST_L2,
    5
)

# Keep only deep interior pixels as high-confidence foreground markers.
_, sure_fg = cv2.threshold(
    dist_transform,
    0.5 * dist_transform.max(),
    255,
    0
)

sure_fg = np.uint8(
    sure_fg
)

# Pixels that are neither sure foreground nor sure background remain unknown.
unknown = cv2.subtract(
    sure_bg,
    sure_fg
)

num_markers, markers = cv2.connectedComponents(
    sure_fg
)

# Reserve label 0 for unknown pixels, as required by OpenCV watershed.
markers = markers + 1
markers[unknown == 255] = 0

markers_ws = cv2.watershed(
    cv2.cvtColor(
        watershed_input,
        cv2.COLOR_RGB2BGR
    ),
    markers.copy()
)

watershed_view = watershed_input.copy()
watershed_view[
    markers_ws == -1
] = [255, 0, 0]

fig, axes = plt.subplots(
    2,
    3,
    figsize=(15, 10)
)

show_rgb(
    axes[0, 0],
    peppers_rgb,
    "Input"
)

show_gray(
    axes[0, 1],
    opening,
    "Opening"
)

show_gray(
    axes[0, 2],
    sure_bg,
    "Sure background"
)

show_gray(
    axes[1, 0],
    dist_transform,
    "Distance transform"
)

show_gray(
    axes[1, 1],
    sure_fg,
    "Sure foreground"
)

show_rgb(
    axes[1, 2],
    watershed_view,
    "Watershed boundaries"
)

fig.tight_layout()
save_figure(
    fig,
    "20_watershed.png"
)
plt.show()


> **Output comment.** Watershed uses foreground/background markers to partition ambiguous touching regions. Its effectiveness depends strongly on marker quality: insufficient markers cause under-segmentation, while excessive markers produce over-segmentation. The method should therefore be interpreted as marker-driven region separation rather than an automatic universal solution.


## 23. Ground Truth and Segmentation Metrics


In [ ]:
def segmentation_metrics(
    ground_truth,
    prediction
):
    """Compute confusion counts and overlap metrics for binary masks.
    
    Parameters
    ----------
    ground_truth, prediction : array-like
        Same-shaped masks converted to Boolean semantics.
    
    Returns
    -------
    dict
        TP, TN, FP, FN, Accuracy, Precision, Recall, IoU and Dice.
    
    Notes
    -----
    IoU and Dice are emphasized because background-dominated pixel accuracy can
    look high even when the foreground object is poorly segmented. A tiny epsilon
    only stabilizes empty-mask denominators.
    """
    gt = np.asarray(
        ground_truth
    ).astype(bool)

    pred = np.asarray(
        prediction
    ).astype(bool)

    tp = np.logical_and(
        gt,
        pred
    ).sum()

    tn = np.logical_and(
        ~gt,
        ~pred
    ).sum()

    fp = np.logical_and(
        ~gt,
        pred
    ).sum()

    fn = np.logical_and(
        gt,
        ~pred
    ).sum()

    # Stabilize metric denominators for empty-set edge cases.
    # Epsilon handles empty-mask edge cases without materially changing non-degenerate scores.
epsilon = 1e-12

    accuracy = (
        (tp + tn)
        / (tp + tn + fp + fn + epsilon)
    )

    precision = (
        tp
        / (tp + fp + epsilon)
    )

    recall = (
        tp
        / (tp + fn + epsilon)
    )

    iou = (
        tp
        / (tp + fp + fn + epsilon)
    )

    dice = (
        2 * tp
        / (2 * tp + fp + fn + epsilon)
    )

    return {
        "TP": int(tp),
        "TN": int(tn),
        "FP": int(fp),
        "FN": int(fn),
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "IoU": iou,
        "Dice": dice,
    }

### Synthetic Ground-Truth Check


In [ ]:
# No official hand.png ground truth is provided; this synthetic case validates metric behavior only.
# Use a controlled perturbed mask to verify metric behavior against known ground truth.
demo_gt = clean_components.copy()

demo_prediction = cv2.erode(
    to_uint8_mask(
        clean_components
    ),
    np.ones(
        (7, 7),
        np.uint8
    ),
    iterations=1
) > 0

demo_metrics = segmentation_metrics(
    demo_gt,
    demo_prediction
)

for key, value in demo_metrics.items():
    # Format continuous metrics separately from integer confusion counts.
    if isinstance(value, float):
        print(
            f"{key:10s}: {value:.4f}"
        )
    else:
        print(
            f"{key:10s}: {value}"
        )


> **Output comment.** Pixel accuracy alone can be misleading when background dominates the image. Precision measures false-positive control, recall measures recovered foreground, and IoU/Dice measure mask overlap directly. Using several metrics together gives a more reliable assessment of segmentation quality than any single score.


## 24. Dice vs IoU Relationship

## 25. Under-Segmentation vs Over-Segmentation

## 26. End-to-End Binary Segmentation Pipeline


In [ ]:
# Compose the validated stages into one reusable segmentation pipeline.
def segment_dark_object(
    image_gray,
    blur_kernel=(5, 5),
    minimum_area=500
):
    """Segment a dark object with an explicit classical pipeline.
    
    Parameters
    ----------
    image_gray : ndarray
        uint8 grayscale input in which the target is darker than background.
    blur_kernel : tuple[int, int]
        Gaussian preprocessing window; default 5x5.
    minimum_area : int
        Smallest connected component retained, in pixels.
    
    Returns
    -------
    dict
        Otsu threshold and intermediate/final masks for inspection.
    
    Notes
    -----
    The pipeline deliberately exposes every assumption: Gaussian smoothing,
    inverted Otsu thresholding, elliptical opening/closing, hole filling and
    8-connected area filtering. minimum_area=500 is appropriate for the supplied
    hand image and must be retuned for substantially different resolutions.
    """
    # The default 5x5 blur stabilizes Otsu without erasing the target boundary.
    blurred = cv2.GaussianBlur(
        image_gray,
        blur_kernel,
        0
    )

    threshold_value, binary = cv2.threshold(
        blurred,
        0,
        255,
        cv2.THRESH_BINARY_INV
        + cv2.THRESH_OTSU
    )

    # A compact 5x5 ellipse regularizes boundaries with limited directional bias.
    kernel = cv2.getStructuringElement(
        cv2.MORPH_ELLIPSE,
        (5, 5)
    )

    cleaned = cv2.morphologyEx(
        binary,
        cv2.MORPH_OPEN,
        kernel
    )

    cleaned = cv2.morphologyEx(
        cleaned,
        cv2.MORPH_CLOSE,
        kernel
    )

    filled = ndimage.binary_fill_holes(
        cleaned > 0
    )

    num_labels, labels, stats, _ = (
        cv2.connectedComponentsWithStats(
            to_uint8_mask(filled),
            # Preserve diagonal connectivity when measuring object components.
            connectivity=8
        )
    )

    final_mask = np.zeros_like(
        filled,
        dtype=bool
    )

    for label_id in range(
        1,
        num_labels
    ):
        area = stats[
            label_id,
            cv2.CC_STAT_AREA
        ]

        # Keep only components large enough to represent the target object.
        # Keep only components that satisfy the explicit object-size prior.
        if area >= minimum_area:
            final_mask |= (
                labels == label_id
            )

    return {
        "threshold": threshold_value,
        "blurred": blurred,
        "binary": binary > 0,
        "cleaned": cleaned > 0,
        "filled": filled,
        "mask": final_mask,
    }

In [ ]:
# Run the complete pipeline once and expose every intermediate stage for inspection.
# Reuse the same 500-pixel area prior from the earlier component-analysis experiment.
hand_result = segment_dark_object(
    hand,
    minimum_area=500
)

final_hand_mask = hand_result[
    "mask"
]

final_overlay = overlay_mask(
    np.stack(
        [hand] * 3,
        axis=-1
    ),
    final_hand_mask
)

fig, axes = plt.subplots(
    2,
    3,
    figsize=(15, 10)
)

show_gray(
    axes[0, 0],
    hand,
    "Input"
)

show_gray(
    axes[0, 1],
    hand_result["blurred"],
    "Smoothed"
)

show_gray(
    axes[0, 2],
    hand_result["binary"],
    "Otsu threshold"
)

show_gray(
    axes[1, 0],
    hand_result["cleaned"],
    "Morphological cleanup"
)

show_gray(
    axes[1, 1],
    final_hand_mask,
    "Final mask"
)

show_rgb(
    axes[1, 2],
    final_overlay,
    "Final overlay"
)

fig.tight_layout()
save_figure(
    fig,
    "21_complete_pipeline.png"
)
plt.show()

print(
    "Pipeline Otsu threshold:",
    hand_result["threshold"]
)


> **Output comment.** The integrated binary pipeline demonstrates that useful segmentation usually emerges from a sequence of complementary steps rather than one threshold alone: preprocessing controls noise, thresholding creates an initial mask, morphology regularizes it, and component analysis enforces region-level assumptions.


## 27. Segmentation Method Selection Criteria

## 28. Integrated Segmentation Workflow

## Final Analysis & Interpretation

### Main findings

- Histogram inspection determines whether one global intensity threshold is plausible; overlap between classes signals the need for additional preprocessing or alternative features.
- Manual thresholding is transparent but sensitive to the chosen cutoff. Otsu removes manual selection for approximately bimodal data, while adaptive thresholding addresses spatially varying illumination at the cost of additional neighborhood parameters.
- Morphological erosion, dilation, opening, closing, and hole filling alter mask structure in controlled ways and must be matched to the scale and geometry of the objects being preserved.
- Connected-component analysis converts a pixel mask into measurable regions, allowing explicit filtering by area and other object-level properties.
- HSV segmentation demonstrates how chromatic separation can succeed when grayscale intensity is insufficient, including the special handling required for red hue wrap-around.
- Edge-based segmentation provides boundary evidence but usually requires linking or morphology before it becomes a filled object mask.
- Distance transforms and watershed provide marker-driven separation of touching regions; their quality depends directly on marker construction.
- Accuracy alone is insufficient for foreground segmentation. Precision, recall, IoU, and Dice provide complementary views of false positives, missed foreground, and overlap.
- The integrated dark-object pipeline makes preprocessing, thresholding, morphology, hole filling, connectivity, and area assumptions explicit and traceable.

### Engineering interpretation

Segmentation quality is determined by the representation and assumptions used before the final mask appears. Threshold choice, local illumination, color space, structuring-element scale, connectivity, region priors, and evaluation metric can each change the result. A robust workflow therefore exposes intermediate masks instead of treating segmentation as one opaque operation.

### Limitations

Several thresholds and geometric priors are tuned to the supplied examples, especially the 500-pixel area criterion and HSV bounds. These values must be reconsidered when image resolution, target size, illumination, or acquisition conditions change.

### Final conclusion

The notebook provides a complete classical segmentation workflow spanning global/local thresholding, morphology, connected regions, color cues, edge evidence, distance transforms, watershed, quantitative metrics, failure-mode analysis, and a validated integrated pipeline.
